This notebook downloads and performs the INFORMED-SKY analysis on GW250114 data.

We assume a ringdown model with 221+220 modes. 

We also calculate the $\ln\mathcal{B}^{221+220}_{220}$

In [1]:
import os

# jax device and parallelization flags on m4 mac.
# modify as needed.
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"

import sys
sys.path.append('../../src/')

import numpy as onp
import jax
import jax.numpy as jnp
from jax.scipy.special import logsumexp

from myutils_corrected import Ntime, ACFs, set_detectors, set_detector_locations
from myutils_corrected import ln_likelihood_full_jit, cov_resp, initialize_det_locs

from scipy.linalg import toeplitz, solve_triangular
import scipy.signal as sig

import lal
from gwpy.timeseries import TimeSeries
from other_utils import bandpass_ds, analysis_data, interp1d_jax, load_tables

jax.config.update("jax_enable_x64", True) 

/Users/kallol/Work/skyRing/examples/informed-sky/../../src/myutils_corrected.py:3: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from corner import corner
from chainconsumer import Chain, ChainConsumer, Truth, ChainConfig, PlotConfig

import h5py

import numpyro
from numpyro.infer import MCMC, NUTS
from numpyro.contrib.nested_sampling import NestedSampler
import numpyro.distributions as dist
numpyro.enable_x64()

/Users/kallol/miniconda3/envs/skyloc_ringdown/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data download parameters

This uses data downloaded directly from gwpy.

Set the parameters to download the strain.

In [3]:
tgps = 1420878141.235932 
tM = 0.337e-3 
tgps = tgps + 6*tM

seglen = 8
fs = 16384
fmin = 30
fmax       = 1500
event_id = "GW250114"
plot_checks = 0
T = 0.2
srate = 4096

ra = 2.33     #right ascension
dec = 0.190   #declination

factor = 10
seed       = 31567
t0         = 0.0

qnm1_path  = "../../data/l2/n1l2m2.dat"
qnm2_path  = "../../data/l2/n2l2m2.dat"

Fetching strain data for H1 and L1

In [4]:
dH1 = TimeSeries.fetch_open_data('H1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)
dL1 = TimeSeries.fetch_open_data('L1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)

Calculate H1 and L1 starting times corresponding the reported sky locations (ra=2.33, dec=0.19). 

Calculate PSDs directly from the data.

In [5]:
n_analyze = Ntime(srate, T)

# Calculate H1 and L1 times

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('H1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].location, ra, dec, tgps))
tH1 = tgps + dt_ifo

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('L1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].location, ra, dec, tgps))
tL1 = tgps + dt_ifo


dH1_cond = bandpass_ds(dH1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)
dL1_cond = bandpass_ds(dL1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)

# Calculate PSDs

nperseg = int(seglen * (1/(dH1.times.value[1] - dH1.times.value[0])))
noverlap = nperseg // 2

psdH = sig.welch(
    dH1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",    
)

psdL = sig.welch(
    dL1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",     
)

In [6]:
print('H1 time: ', tH1)
print('L1 time: ', tL1)

H1 time:  1420878141.221026
L1 time:  1420878141.2185555


In [7]:
if tH1>tL1:
    ref_det = 'H1'
    print('H1 peak appears later')
else:
    ref_det = 'L1'
    print('L1 peak appears later')

H1 peak appears later


Set priors for the PE

In [8]:
# Prior limits
limits = [
    [55.0, 95.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0., 3.141592653589793]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

Interpolating PSDs over freq range, setting up cov (using Cholesky decomposition approach) for TD likelihood

In [9]:
psdH = interp1d_jax(jnp.asarray(psdH[0],dtype=jnp.float64), jnp.asarray(psdH[1],dtype=jnp.float64))
psdL = interp1d_jax(jnp.asarray(psdL[0],dtype=jnp.float64), jnp.asarray(psdL[1],dtype=jnp.float64))

omega_r, omega_i, omega_OT_r, omega_OT_i = load_tables(qnm1_path, qnm2_path)
tgps = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps)

L_H, L_L, resp_H, resp_L = cov_resp(srate=srate, T=T, psdH=psdH, psdL=psdL, factor=factor)
set_detectors(resp_H, resp_L)

rH, rL = initialize_det_locs()
set_detector_locations(rH, rL)

In [10]:
# Data strain starting from tH1 in both H1 and L1 for the Full-sky analysis since tH1>tL1
dH1_analysis_data = analysis_data(dH1_cond[1], dH1_cond[0], tH1, n_analyze)
dL1_analysis_data = analysis_data(dL1_cond[1], dL1_cond[0], tH1, n_analyze)

h_H_t = dH1_analysis_data[1]
h_L_t = dL1_analysis_data[1]

We obtain IMR posteriors from open data in GWOSC.

In [12]:
data_dir = "../posterior_files/"

with h5py.File("../posterior_files/GW250114_posterior_samples_NRSur7dq4.h5", "r") as f:
    post_samples = f['bilby-NRSur7dq4_prod-reweighted']['posterior_samples'][:]

cosi_imr = onp.cos(post_samples['iota'])
ra_imr = post_samples['ra']
sindec_imr = onp.sin(post_samples['dec'])
psi_imr = post_samples["psi"]


ext_post = onp.array([ra_imr, sindec_imr]).T

logl_post = jnp.array(post_samples['log_likelihood'])
logp_post = jnp.array(post_samples['log_prior'])
log_post = logl_post + logp_post

We split the parameter set into:
- $\theta$: parameters sampled in the analysis
- $\phi = (\mathrm{ra}, \sin\delta)$: nuisance sky-location parameters marginalized over using IMR posterior samples.

The nuisance-marginalized likelihood is

$$
p(d \mid \theta) = \int p(d \mid \theta,\phi)\, p(\phi)\, d\phi .
$$

This integral is approximated using importance sampling:

$$
p(d \mid \theta) \approx \sum_i p(d \mid \theta,\phi_i)\, w_i ,
$$

where the quadrature points are drawn from the IMR posterior

$$
\phi_i \sim q(\phi)=p_{\rm IMR}(\phi \mid d_{\rm IMR}) .
$$

The corresponding importance weights are

$$
w_i \propto \frac{p(\phi_i)}{q(\phi_i)} .
$$

Using Bayes' theorem,

$$
q(\phi)=p_{\rm IMR}(\phi \mid d_{\rm IMR}) \propto p_{\rm IMR}(d_{\rm IMR} \mid \phi)\, p(\phi),
$$

so that

$$
w_i \propto \frac{p(\phi_i)}{p_{\rm IMR}(d_{\rm IMR} \mid \phi_i)p(\phi_i)} \propto \frac{1}{p_{\rm IMR}(d_{\rm IMR} \mid \phi_i)} .
$$

The likelihood used in the sampler is therefore

$$
\log p(d \mid \theta) \approx \log \sum_i p(d \mid \theta,\phi_i)\, w_i .
$$

In [13]:
# 100 random samples drawn from the IMR posterior
nsamp = 100
idxs = onp.random.randint(low=0, high=len(ext_post), size=nsamp)
ext_mc_integ = ext_post[idxs]
log_post_mc_integ = log_post[idxs]

In [14]:
# Calculate weights as described
def make_logw_normalized(log_p_phi, log_q_phi):
    logw = log_p_phi - log_q_phi                     
    logw = logw - logsumexp(logw)                    
    return logw                                      

logw = make_logw_normalized(0.0, logl_post)
logw_mc_integ = logw[idxs]

Setup the numpyro model for PE

In [ ]:
def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                   omega_r, omega_i, omega_OT_r, omega_OT_i,
                   T, srate, t0):
    def _loglik(theta):
        return ln_likelihood_full_jit(
            dataH=dataH, dataL=dataL, params=theta,
            gmst=gmst, L_H=L_H, L_L=L_L,
            omega_r=omega_r, omega_i=omega_i,
            omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
            T=T, srate=srate, t0=t0, ref_det=ref_det
        )
    return _loglik

loglik_fn = make_loglik_fn(
    dataH=h_H_t, dataL=h_L_t,
    gmst=gmst, L_H=L_H, L_L=L_L,
    omega_r=omega_r, omega_i=omega_i,
    omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
    T=T, srate=srate, t0=t0
)

def loglike(theta_free):
    theta_free_batched = jnp.broadcast_to(theta_free, (nsamp, theta_free.shape[0]))  

    thetas_batched = jnp.concatenate(
                                    [
                                        theta_free_batched[:, :7],          
                                        ext_mc_integ,             
                                        theta_free_batched[:, 7:8],       
                                    ],
                                    axis=1,
                                )

    logliks = jax.vmap(loglik_fn)(thetas_batched)
    return logsumexp(logliks + logw_mc_integ)

In [27]:
low  = jnp.asarray(low,  dtype=jnp.float64)
high = jnp.asarray(high, dtype=jnp.float64)

def model_informed_sky():
    theta_free = numpyro.sample("theta", dist.Uniform(low, high))  
    numpyro.factor("loglike", loglike(theta_free))


In [ ]:
rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
rng_run, rng_post = jax.random.split(rng_key)
ns_informed_sky = NestedSampler(
    model_informed_sky,
    constructor_kwargs=dict(
        num_live_points=10000,    
        max_samples=500_000,      
        verbose=False,
    ),
    termination_kwargs=dict(
        dlogZ=0.01,               
    ),
)
ns_informed_sky.run(rng_run)
ns_informed_sky.print_summary()
posterior_informed_sky = ns_informed_sky.get_samples(rng_post, num_samples=100_000)

In [29]:
ns_informed_sky._results.log_Z_mean

Array(-740.4285773, dtype=float64)

In [ ]:
posterior_dir = '../posterior_files/informed-sky/'
onp.save(posterior_dir + 'posterior.' + event_id + '.informed-sky.npy', onp.asarray(posterior_informed_sky['theta']))

Now doing the same as above for the 220 model to calculate the Bayes factor

In [36]:
limits = [
    [55.0, 95.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0., 3.141592653589793]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

Setup the numpyro model for PE

In [ ]:
def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                   omega_r, omega_i, omega_OT_r, omega_OT_i,
                   T, srate, t0):
    def _loglik(theta):
        return ln_likelihood_full_jit(
            dataH=dataH, dataL=dataL, params=theta,
            gmst=gmst, L_H=L_H, L_L=L_L,
            omega_r=omega_r, omega_i=omega_i,
            omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
            T=T, srate=srate, t0=t0, ref_det=ref_det
        )
    return _loglik

loglik_fn = make_loglik_fn(
    dataH=h_H_t, dataL=h_L_t,
    gmst=gmst, L_H=L_H, L_L=L_L,
    omega_r=omega_r, omega_i=omega_i,
    omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
    T=T, srate=srate, t0=t0
)

def loglike_220(theta_free):
    theta_free_batched = jnp.broadcast_to(theta_free, (nsamp, theta_free.shape[0]))  
    amp_phase_221_batched = jnp.broadcast_to(jnp.array([0.,0.]), (nsamp, 2))
    thetas_batched = jnp.concatenate(
                        [
                            theta_free_batched[:, :4],
                            amp_phase_221_batched,
                            theta_free_batched[:,4:5],
                            ext_mc_integ,              
                            theta_free_batched[:, 5:6],          
                        ],
                        axis=1,
                    )

    logliks = jax.vmap(loglik_fn)(thetas_batched)
    return logsumexp(logliks + logw_mc_integ)

In [39]:
def model_informed_sky_220():
    theta_free = numpyro.sample("theta", dist.Uniform(low, high))
    numpyro.factor("loglike", loglike_220(theta_free))

In [ ]:
rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
rng_run, rng_post = jax.random.split(rng_key)
ns_informed_sky_220 = NestedSampler(
    model_informed_sky_220,
    constructor_kwargs=dict(
        num_live_points=10000,    
        max_samples=500_000,   
        verbose=False,
    ),
    termination_kwargs=dict(
        dlogZ=0.01,             
    ),
)
ns_informed_sky_220.run(rng_run)
ns_informed_sky_220.print_summary()
posterior_informed_sky_220 = ns_informed_sky_220.get_samples(rng_post, num_samples=100_000)

In [41]:
ns_informed_sky_220._results.log_Z_mean

Array(-745.54485204, dtype=float64)

In [42]:
ns_informed_sky._results.log_Z_mean - ns_informed_sky_220._results.log_Z_mean

Array(5.11627474, dtype=float64)